# Apache Spark for AI Data Engineering

**Phase:** Data & Systems Foundations  
**Prerequisites:** NB06 (Data Lakehouse), NB07 (Data Management)  
**Toolchain:** PySpark 4.1 · Spark SQL · Spark Declarative Pipelines (SDP) · PyArrow  
**Objective:** Master distributed data processing with Spark — from DataFrame API fundamentals through production-grade Medallion pipelines, Spark SQL, partitioning strategies, and real-time streaming.

---

## Why Spark Before ML?

When your dataset exceeds ~50GB, Pandas collapses with `MemoryError`. Production ML pipelines routinely process terabytes of training data. Spark distributes this work across a cluster of machines, turning hours into minutes.

| Data Size | Tool | Why |
|:-:|---|---|
| < 1 GB | Pandas | Fast, simple, in-memory |
| 1–50 GB | Polars / DuckDB | Single-machine but optimized |
| 50 GB–PB | **Apache Spark** | Distributed across cluster |

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** When data exceeds 50GB, Pandas completely collapses (MemoryError). Seniors leverage distributed processing frameworks like Spark to crunch terabytes horizontally across clusters.

**Mental Model:** Pandas is a single powerful chef in a small kitchen. Spark is an executive chef (Driver) orchestrating 100 sous-chefs (Executors) working on pieces of the meal simultaneously.

**Common Junior Pitfall:** Calling `.collect()` on a massive Spark DataFrame, pulling 100GB of data back into the single Driver node, crashing the entire Spark application.

---


## 📑 Table of Contents

* [Why Spark Before ML?](#why-spark-before-ml)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Spark Core Architecture](#1--spark-core-architecture)
* [2 · DataFrame API Fundamentals](#2--dataframe-api-fundamentals)
  * [Pandas vs PySpark Translation](#pandas-vs-pyspark-translation)
* [3 · Spark SQL: Querying Distributed Data](#3--spark-sql-querying-distributed-data)
* [4 · Joins, Broadcast & Partitioning](#4--joins-broadcast--partitioning)
* [5 · UDFs: Python → PyArrow Performance](#5--udfs-python--pyarrow-performance)
* [6 · Medallion Architecture Pipeline](#6--medallion-architecture-pipeline)
* [7 · Spark Declarative Pipelines (SDP)](#7--spark-declarative-pipelines-sdp)
* [8 · Structured Streaming & RTM](#8--structured-streaming--rtm)
* [9 · Performance Tuning Checklist](#9--performance-tuning-checklist)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Practice](#-practical-practice)
* [🎯 Summary & Key Takeaways](#-summary--key-takeaways)


---
## 1 · Spark Core Architecture

```
Your PySpark Code
       |
  [Driver] — orchestrates, holds SparkSession, runs main()
       |
  [Cluster Manager] — allocates resources (YARN, K8s, Standalone)
      / | \
 [Executor] [Executor] [Executor]  — worker JVMs processing data
   |          |          |
 [Task]     [Task]     [Task]      — unit of work on one partition
```

### Key Concepts

| Concept | What It Means | Why It Matters |
|---------|--------------|----------------|
| **Lazy evaluation** | Transformations build a plan, actions execute it | Catalyst optimizer can rearrange operations |
| **Partition** | A chunk of data processed by one task | Too few = idle cores; too many = scheduling overhead |
| **Shuffle** | Data redistribution across executors | Most expensive operation — avoid when possible |
| **Catalyst** | SQL query optimizer | Rewrites your plan for optimal execution |
| **Tungsten** | Memory management engine | Off-heap memory, binary encoding, code generation |

### Transformations vs Actions

| Transformations (Lazy) | Actions (Trigger Execution) |
|---|---|
| `select`, `filter`, `groupBy` | `show`, `count`, `collect` |
| `join`, `withColumn`, `agg` | `write`, `take`, `first` |
| Build the DAG plan | Execute the DAG plan |


In [ ]:
# Cell 1 — Initialize Spark and explore the session
!pip install -q pyspark pandas pyarrow numpy


In [ ]:
# Cell 2 — Spark session setup
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AI_Data_Engineering") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} running locally")
print(f"   Cores: {spark.sparkContext.defaultParallelism}")
print(f"   AQE (Adaptive Query Execution): enabled")


---
## 2 · DataFrame API Fundamentals

The DataFrame API is Spark's primary interface. If you know Pandas, the mental translation is straightforward:

### Pandas vs PySpark Translation

| Operation | Pandas | PySpark |
|-----------|--------|--------|
| Read CSV | `pd.read_csv('f.csv')` | `spark.read.csv('f.csv', header=True, inferSchema=True)` |
| Select columns | `df[['a', 'b']]` | `df.select('a', 'b')` |
| Filter rows | `df[df.x > 5]` | `df.filter(col('x') > 5)` |
| New column | `df['y'] = df.x * 2` | `df.withColumn('y', col('x') * 2)` |
| Group + agg | `df.groupby('a').mean()` | `df.groupBy('a').agg(avg('x'))` |
| Sort | `df.sort_values('x')` | `df.orderBy('x')` |
| Drop NaN | `df.dropna()` | `df.dropna()` |
| Shape | `df.shape` | `(df.count(), len(df.columns))` |
| Preview | `df.head()` | `df.show(5)` |

### ⚠️ The `.collect()` Rule

**Never call `.collect()` on large DataFrames.** It pulls all data to the Driver's memory. Use:
- `.show(n)` to preview rows
- `.take(n)` to get a small list
- `.write` to save results to storage


In [ ]:
# Cell 3 — DataFrame API fundamentals
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, avg, count, sum as spark_sum, when, lit, round as spark_round

# Create a realistic ML training dataset
np.random.seed(42)
n = 10000
pdf = pd.DataFrame({
    'user_id': [f'user_{i}' for i in range(n)],
    'age': np.random.normal(35, 12, n).astype(int).clip(18, 80),
    'income': np.random.lognormal(10.5, 0.8, n).round(2),
    'category': np.random.choice(['electronics', 'clothing', 'food', 'books', 'sports'], n),
    'purchase_amount': np.random.exponential(50, n).round(2),
    'is_premium': np.random.choice([0, 1], n, p=[0.7, 0.3]),
})

# Load into Spark (in production: spark.read.parquet('s3://...'))
df = spark.createDataFrame(pdf)
print(f'DataFrame: {df.count()} rows x {len(df.columns)} columns')
print(f'Partitions: {df.rdd.getNumPartitions()}')
df.printSchema()

# === Core Operations ===

# 1. Select + filter
premium_users = df.filter(col('is_premium') == 1).select('user_id', 'income', 'purchase_amount')
print(f'\nPremium users: {premium_users.count()}')

# 2. withColumn (add computed columns)
df_enriched = df.withColumn('income_bracket',
    when(col('income') < 30000, 'low')
    .when(col('income') < 80000, 'medium')
    .otherwise('high')
)
df_enriched.groupBy('income_bracket').count().show()

# 3. GroupBy + Aggregation
category_stats = df.groupBy('category').agg(
    spark_round(avg('purchase_amount'), 2).alias('avg_purchase'),
    spark_round(avg('income'), 0).alias('avg_income'),
    count('*').alias('user_count'),
    spark_sum('purchase_amount').alias('total_revenue'),
).orderBy('total_revenue', ascending=False)

print('Category Statistics:')
category_stats.show()


---
## 3 · Spark SQL: Querying Distributed Data

Spark SQL lets you write standard SQL against distributed DataFrames. This is powerful because:
1. Many data engineers are more fluent in SQL than Python
2. SQL queries go through the **same Catalyst optimizer** as DataFrame API
3. You can query across **Delta Lake tables, Parquet files, and CSVs** with one syntax


In [ ]:
# Cell 4 — Spark SQL

# Register DataFrame as a temporary SQL view
df.createOrReplaceTempView('users')

# Now query with SQL!
result = spark.sql("""
    SELECT 
        category,
        COUNT(*) as user_count,
        ROUND(AVG(purchase_amount), 2) as avg_purchase,
        ROUND(SUM(purchase_amount), 2) as total_revenue,
        ROUND(AVG(CASE WHEN is_premium = 1 THEN purchase_amount END), 2) as avg_premium_purchase
    FROM users
    WHERE age BETWEEN 25 AND 45
    GROUP BY category
    HAVING COUNT(*) > 100
    ORDER BY total_revenue DESC
""")

print('Spark SQL — Category Revenue (ages 25-45):')
result.show()

# Window functions (advanced SQL that Pandas can't do as cleanly)
windowed = spark.sql("""
    SELECT 
        user_id, category, purchase_amount,
        RANK() OVER (PARTITION BY category ORDER BY purchase_amount DESC) as rank_in_category,
        ROUND(purchase_amount / SUM(purchase_amount) OVER (PARTITION BY category), 4) as pct_of_category
    FROM users
""")
print('Window functions — Top spenders per category:')
windowed.filter(col('rank_in_category') <= 3).orderBy('category', 'rank_in_category').show(15)

print('💡 Both DataFrame API and SQL produce the same optimized physical plan.')
print('   Use whichever your team prefers — performance is identical.')


---
## 4 · Joins, Broadcast & Partitioning

### The Shuffle Problem

Joins require matching keys across partitions. If data is randomly distributed, Spark must **shuffle** — redistribute data across the network. This is the single most expensive operation in Spark.

```
Without broadcast:                    With broadcast:
Executor 1 ←→ Executor 2             Executor 1 (has full lookup table)
Executor 2 ←→ Executor 3             Executor 2 (has full lookup table)
Executor 3 ←→ Executor 1             Executor 3 (has full lookup table)
(shuffle across network)              (no shuffle needed!)
```

### Join Strategies

| Strategy | When | Performance |
|----------|------|------------|
| **Broadcast join** | One table < 100MB | ⚡ Fastest (no shuffle) |
| **Sort-merge join** | Both tables large, join keys sortable | Standard for big+big |
| **Shuffle hash join** | One table much smaller (but >100MB) | Good for skewed data |

### Partitioning Strategies

| Strategy | Use Case | Example |
|----------|----------|--------|
| **Hash partitioning** | Even distribution by key | `df.repartition(200, 'user_id')` |
| **Range partitioning** | Sorted access patterns | `df.repartitionByRange(200, 'date')` |
| **Coalesce** | Reduce partitions (no shuffle) | `df.coalesce(10)` after filtering |

### ⚠️ Partition Count Rule of Thumb

- **Target:** 100–200 MB per partition
- **Formula:** `num_partitions = total_data_size / 128MB`
- Too few partitions → OOM errors, idle cores
- Too many partitions → scheduling overhead, tiny tasks


In [ ]:
# Cell 5 — Joins and broadcast
from pyspark.sql.functions import broadcast

# Create a small lookup table (category metadata)
category_info = spark.createDataFrame([
    ('electronics', 'Tech', 0.15),
    ('clothing', 'Fashion', 0.08),
    ('food', 'Grocery', 0.05),
    ('books', 'Media', 0.10),
    ('sports', 'Lifestyle', 0.12),
], ['category', 'department', 'commission_rate'])

# === Regular join (Spark may or may not broadcast automatically) ===
joined = df.join(category_info, on='category', how='left')

# === Explicit broadcast join (FORCE no-shuffle) ===
joined_fast = df.join(broadcast(category_info), on='category', how='left')

# Add computed column using joined data
with_commission = joined_fast.withColumn(
    'commission', spark_round(col('purchase_amount') * col('commission_rate'), 2)
)

print('Broadcast Join Result (no shuffle!):')
with_commission.select('user_id', 'category', 'department', 'purchase_amount', 'commission').show(5)

# Show the physical plan — notice 'BroadcastHashJoin'
print('Physical Plan (look for BroadcastHashJoin):')
joined_fast.explain()

# === Partitioning demo ===
print(f'\nPartitions before: {df.rdd.getNumPartitions()}')
df_repartitioned = df.repartition(8, 'category')  # Hash partition by category
print(f'Partitions after repartition(8, category): {df_repartitioned.rdd.getNumPartitions()}')

# After a filter that reduces data significantly, coalesce to reduce partitions
df_filtered = df.filter(col('is_premium') == 1)  # ~30% of data
df_coalesced = df_filtered.coalesce(4)  # Reduce without shuffle
print(f'After filter + coalesce: {df_coalesced.rdd.getNumPartitions()} partitions')
print('  → coalesce() merges partitions WITHOUT a shuffle (unlike repartition)')


---
## 5 · UDFs: Python → PyArrow Performance

When Spark's built-in functions aren't enough, you need User Defined Functions (UDFs). But the implementation choice determines a **1000x performance difference**:

| UDF Type | Speed | How It Works |
|----------|-------|-------------|
| Python UDF | 🔴 Slow | Serializes each row JVM → Python → JVM |
| Pandas UDF | 🟡 Fast | Operates on Pandas Series batches |
| PyArrow UDF | 🟢 Fastest | Columnar batches, zero-copy when possible |
| Built-in function | ⚡ Native | Runs inside JVM, no Python overhead |

**Rule:** Always try built-in functions first. Only use UDFs when no built-in exists.


In [ ]:
# Cell 6 — UDF performance comparison
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import FloatType
import pyarrow as pa
import numpy as np
import time

# === 1. Python UDF (SLOW — row-by-row serialization) ===
@udf(FloatType())
def zscore_python(temp, mean_val, std_val):
    return float((temp - mean_val) / (std_val + 1e-10))

# === 2. Pandas UDF (FAST — vectorized batches) ===
@pandas_udf(FloatType())
def zscore_pandas(temp: pd.Series) -> pd.Series:
    return (temp - temp.mean()) / (temp.std() + 1e-10)

# === 3. Built-in functions (FASTEST — no Python overhead) ===
from pyspark.sql.functions import mean as spark_mean, stddev as spark_stddev

# Demo with our DataFrame
print('UDF Performance Comparison:')
print('  Python UDF:   ~500 rows/sec   (serializes each row)')
print('  Pandas UDF:   ~200,000 rows/sec (vectorized batches)')
print('  PyArrow UDF:  ~500,000 rows/sec (columnar, zero-copy)')
print('  Built-in:     ~5,000,000 rows/sec (native JVM execution)')
print()

# PyArrow batch function demo
def compute_zscore_pyarrow(temperature_array: pa.Array) -> pa.Array:
    """PyArrow-native UDF: operates on columnar data."""
    values = temperature_array.to_numpy(zero_copy_only=False)
    mean, std = values.mean(), values.std()
    zscores = (values - mean) / (std + 1e-10)
    return pa.array(zscores, type=pa.float64())

temps = pa.array(np.random.normal(72, 8, 1000))
zscores = compute_zscore_pyarrow(temps)
print(f'PyArrow demo: {len(temps)} temps → {sum(1 for z in zscores.to_pylist() if abs(z) > 2)} anomalies (|z|>2)')
print()
print('💡 Rule: Always check if a built-in Spark function exists before writing a UDF.')
print('   Example: use F.abs(), F.round(), F.when() instead of UDFs.')


---
## 6 · Medallion Architecture Pipeline

The **Medallion Architecture** (from NB06) structures data into quality tiers:

```
Raw Source → [Bronze: Raw Ingestion] → [Silver: Cleaned] → [Gold: ML Features]
```

| Layer | Purpose | Quality | Schema |
|-------|---------|---------|--------|
| Bronze | Raw ingest, append-only | As-is | Schema-on-read |
| Silver | Deduped, typed, validated | Clean | Enforced |
| Gold | Aggregated, ML-ready features | Curated | Optimized |


In [ ]:
# Cell 7 — Full Medallion pipeline in PySpark
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, avg, count, current_timestamp, when, regexp_replace, trim, upper
)

# === BRONZE: Raw Ingestion ===
def bronze_ingest() -> DataFrame:
    """Simulate raw data ingestion (production: read from Kafka/S3)."""
    print('[Pipeline] 1. Bronze: Ingesting raw data...')
    raw = spark.range(0, 100000) \
        .withColumn('device_id', (col('id') % 10).cast('string')) \
        .withColumn('temperature', (col('id') % 150) - 20) \
        .withColumn('quality', (col('id') % 100) / 100.0) \
        .withColumn('timestamp', current_timestamp())
    return raw

# === SILVER: Clean & Validate ===
def silver_clean(bronze_df: DataFrame) -> DataFrame:
    """Apply data quality rules."""
    print('[Pipeline] 2. Silver: Cleaning and validating...')
    return (
        bronze_df
        .filter(col('device_id').isNotNull())
        .filter(col('temperature').between(-50, 150))  # Physical bounds
        .filter(col('quality') >= 0.5)                  # Quality gate
        .dropDuplicates(['device_id', 'timestamp'])
    )

# === GOLD: ML Feature Aggregation ===
def gold_features(silver_df: DataFrame) -> DataFrame:
    """Build ML-ready features."""
    print('[Pipeline] 3. Gold: Aggregating ML features...')
    return (
        silver_df
        .groupBy('device_id')
        .agg(
            avg('temperature').alias('avg_temp'),
            count('*').alias('reading_count'),
            avg('quality').alias('avg_quality'),
        )
        .withColumn('health_score',
            when(col('avg_quality') > 0.8, 'healthy')
            .when(col('avg_quality') > 0.6, 'degraded')
            .otherwise('critical')
        )
        .orderBy('device_id')
    )

# === Execute the pipeline ===
bronze = bronze_ingest()
silver = silver_clean(bronze)
gold = gold_features(silver)

print(f'\nBronze rows: {bronze.count()}')
print(f'Silver rows: {silver.count()} (after quality filter)')
print(f'Gold rows:   {gold.count()} (device-level features)')
print('\n--- Gold Layer Output ---')
gold.show()

# Show the Catalyst optimizer's physical plan
print('--- Catalyst Physical Plan ---')
gold.explain()


---
## 7 · Spark Declarative Pipelines (SDP)

Spark 4.1 introduced **Declarative Pipelines** — instead of managing execution order manually, you declare *what* each table should contain, and Spark handles *how* to build it.

### Imperative vs Declarative

```python
# IMPERATIVE (old): You manage order, retries, checkpoints
df = spark.read.parquet('s3://raw/')
df = df.filter(df.quality > 0.8)
df.write.mode('overwrite').parquet('s3://gold/')

# DECLARATIVE (new): Spark manages everything
@dp.table
def gold_prices():
    return spark.read.table('silver').groupBy('cat').agg(avg('price'))
# Spark auto-builds DAG, handles retries, checkpointing
```

### Trade-offs

| | SDP (Declarative) | Traditional (Imperative) |
|---|---|---|
| Control | Less (Spark decides) | Full (you decide) |
| Debugging | Harder (abstracted) | Easier (step-by-step) |
| Best for | Standard ETL | Complex custom logic |
| Auto-retry | Built-in | Manual |


---
## 8 · Structured Streaming & RTM

Traditional Spark Streaming uses **micro-batching** (1-5 second intervals). Spark 4.1's **Real-Time Mode (RTM)** processes records immediately.

| Mode | Latency | Throughput | Use Case |
|------|---------|-----------|----------|
| Micro-batch | 1–10 sec | Very high | Analytics, hourly reports |
| RTM | 1–50 ms | High | Fraud detection, live features |

### Spark Streaming vs Flink

| | Spark RTM | Apache Flink |
|---|---|---|
| Latency | 1–50ms | 1–10ms |
| Ecosystem | Full Spark SQL, MLlib, Delta | Flink SQL only |
| Best for | Teams already on Spark | Pure streaming workloads |

> **Decision rule:** If your team already uses Spark for batch, use RTM for streaming (one stack). If you need sub-10ms latency for millions of events/sec, evaluate Flink (NB10).


---
## 9 · Performance Tuning Checklist

| Problem | Symptom | Fix |
|---------|---------|-----|
| **Data skew** | One task takes 10x longer | Salting keys, AQE skew join |
| **Too few partitions** | OOM, low parallelism | `repartition(n)` |
| **Too many partitions** | High scheduler overhead | `coalesce(n)` |
| **Shuffle on join** | Slow join performance | `broadcast()` small table |
| **Python UDF** | 1000x slower than needed | Use built-in or Pandas UDF |
| **collect() on big data** | Driver OOM crash | Use `.show()` or `.write()` |
| **Small files** | Slow reads | Compaction, `coalesce` before write |

### Memory Configuration

```python
# Production Spark config for ML data processing
spark = SparkSession.builder \
    .config('spark.executor.memory', '8g') \
    .config('spark.executor.cores', '4') \
    .config('spark.sql.shuffle.partitions', '200') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer') \
    .getOrCreate()
```


---
## ✅ Knowledge Check

### Q1: Lazy Evaluation
You write `df = spark.read.parquet('s3://data/')` then `df = df.filter(df.x > 5)`. Has Spark read any data?

<details><summary>👉 Answer</summary>

**No.** Transformations like `filter`, `select`, `groupBy` are recorded in the DAG but NOT executed. Data is only processed when you call an **action** like `.show()`, `.count()`, or `.write()`. This lets Catalyst optimize the entire plan first.
</details>

### Q2: Broadcast Join
You're joining a 500GB user table with a 50MB country lookup. What join strategy should you use?

<details><summary>👉 Answer</summary>

**Broadcast join.** The 50MB table is small enough to fit in each executor's memory. Use `df.join(broadcast(country_table), ...)` to send the small table to all executors, eliminating the expensive shuffle of the 500GB table.
</details>

### Q3: Partition Count
You have 100GB of Parquet data. How many partitions should you target?

<details><summary>👉 Answer</summary>

**~800 partitions.** Rule of thumb: target 100–200MB per partition. 100GB / 128MB ≈ 800. Too few → OOM; too many → scheduling overhead.
</details>

### Q4: PyArrow UDFs
Why are PyArrow UDFs ~1000x faster than regular Python UDFs?

<details><summary>👉 Answer</summary>

Python UDFs serialize each row individually from JVM to Python and back. PyArrow UDFs operate on **columnar batches** — entire columns passed as Arrow arrays without per-row serialization. This eliminates the JVM-Python border crossing bottleneck.
</details>


---
## 🔨 Practical Practice

### Exercise 1: Pandas to PySpark Translation
Translate this Pandas code to PySpark:
```python
result = df[df['status'] == 'completed'].groupby('user_id')['amount'].sum().reset_index()
```

### Exercise 2: Broadcast Optimization
You have a 200GB orders table and a 20MB product catalog. Write a join that avoids shuffling the orders table.

### Exercise 3: End-to-End ML Data Pipeline
Build a complete Bronze→Silver→Gold pipeline that:
1. Bronze: Reads raw CSV data
2. Silver: Removes nulls, deduplicates, casts types
3. Gold: Computes per-user features (avg purchase, total spend, purchase frequency)
4. Writes Gold layer to Parquet partitioned by date

### Exercise 4: Partition Tuning
Create a 1M row DataFrame, repartition to 4, 40, and 400 partitions. Time a `groupBy + count` on each. Which is fastest?


In [ ]:
# Cell 8 — Exercise Solutions
from pyspark.sql.functions import col, broadcast

print('Ex 1: PySpark Translation')
print("""from pyspark.sql.functions import col, sum as spark_sum
result = (
    df.filter(col('status') == 'completed')
    .groupBy('user_id')
    .agg(spark_sum('amount').alias('amount'))
)""")

print('\nEx 2: Broadcast Join')
print("""enriched_orders = orders.join(
    broadcast(product_catalog),  # Force broadcast of small table
    on='product_id',
    how='left'
)""")
print('  The broadcast() hint sends the 20MB catalog to every executor.')
print('  The 200GB orders table stays in place — zero shuffle.')

# Cleanup
spark.stop()
print('\n✅ Spark session stopped.')


---
## 🎯 Summary & Key Takeaways

| Concept | What You Learned |
|---------|------------------|
| Architecture | Driver/Executor model, lazy evaluation, DAG |
| DataFrame API | select, filter, groupBy, withColumn, join |
| Spark SQL | SQL queries on distributed data, window functions |
| Joins | Broadcast for small tables, sort-merge for large |
| Partitioning | repartition vs coalesce, 100-200MB per partition |
| UDFs | Built-in > PyArrow > Pandas > Python (1000x diff) |
| Medallion | Bronze → Silver → Gold pipeline pattern |
| SDP | Declarative pipelines with auto-DAG |
| RTM | Sub-50ms streaming in Spark |
| Tuning | AQE, broadcast joins, partition sizing |

**Connections:** Lakehouse tables → NB06 (Delta Lake) | Data validation → NB07 (Great Expectations) | Orchestration → NB09 (Airflow/Dagster) | Feature engineering → NB11 (Feast)

**Next →** `09_data_orchestration_pipelines.ipynb` — Pipeline Orchestration with Airflow & Dagster
